# 10. MMPA 기반 치환 라이브러리 자동 확장

## 이번 노트북에서 할 것
- Tox21 데이터에서 "같은 뼈대(core)를 공유하지만 독성 라벨(y)이 다른 분자 쌍"을
  MMPA로 자동 탐색
- 판단 기준은 반드시 Tox21 실제 라벨(y)로 함 — toxicophore_detector 결과를
  기준으로 쓰면 순환논리가 되므로 사용하지 않음
- 찾은 치환쌍을 replacement_library.py에 데이터 기반 candidate로 추가하는 방법 설계

## 간략한 정리 (09까지)
- candidate 선택도 LLM 판단으로 확장 완료 (ask_llm_which_candidate_to_use)
- 중대 버그 발견/수정: find_core_and_target()의 best_match(차선책) 로직이
  화학적으로 무의미한 결과(N#CF)를 만들어냄 → 제거, 정확히 매칭 안되면 None 반환하도록 수정
- Gemini 3.5 Flash 무료 티어 일일 한도(20회) 소진 → 전체 통합 루프 최종 재검증은
  내일 한도 리셋 후 진행 예정 (코드는 완성됨)
- MMPA(rdMMPA.FragmentMol)는 기존에 "재조립"(같은 core, 다른 candidate 붙이기) 용도로만
  썼는데, 이번엔 "라이브러리 자동 확장"(데이터에서 치환쌍 발견) 용도로 재사용

## 다음에 해야 할 것 (오늘 끝나면)
- (내일) Gemini 한도 리셋 후 전체 통합 루프(문제선택+후보선택 LLM화) 최종 검증
- 문헌/FDA 가이드라인 기반 치환쌍도 몇 개 추가 조사 (데이터 기반과 지식 기반 상호보완)
- Held-out set 전체에 루프 적용 → 성공률/개선율 통계
- 제안서(hwpx) 작성 시작 (마감 8/7)

In [1]:
# 셀 1
!pip install rdkit -q

In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix

data = load_tox21_clean()
print("도구 로드 확인 완료")

[11:03:31] WARNING: not removing hydrogen atom without neighbors
[11:03:31] Explicit valence for atom # 8 Al, 6, is greater than permitted
[11:03:31] Explicit valence for atom # 3 Al, 6, is greater than permitted
[11:03:31] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:03:32] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:03:33] Explicit valence for atom # 9 Al, 6, is greater than permitted
[11:03:33] Explicit valence for atom # 5 Al, 6, is greater than permitted
[11:03:33] Explicit valence for atom # 16 Al, 6, is greater than permitted
[11:03:34] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[11:03:34] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [5]:
from collections import defaultdict

def get_core_to_smiles_map(smiles_list, sample_size=500):
    """분자들을 1-cut MMPA로 쪼개서, core를 key로 원본 분자들을 그룹핑."""
    core_map = defaultdict(list)
    for s in smiles_list[:sample_size]:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            continue
        try:
            fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
        except Exception:
            continue
        for core, chain in fragments:
            if core:
                continue
            parts = chain.split('.')
            if len(parts) != 2:
                continue
            p0_mol = Chem.MolFromSmiles(parts[0].replace('[*:1]', '[H]'))
            p1_mol = Chem.MolFromSmiles(parts[1].replace('[*:1]', '[H]'))
            if p0_mol is None or p1_mol is None:
                continue
            if p0_mol.GetNumHeavyAtoms() >= p1_mol.GetNumHeavyAtoms():
                big, small = parts[0], parts[1]
            else:
                big, small = parts[1], parts[0]
            core_map[big].append((s, small))
    return core_map

core_map = get_core_to_smiles_map(list(data['smiles_train']), sample_size=500)
print("고유 core 개수:", len(core_map))

shared = {k: v for k, v in core_map.items() if len(v) >= 2}
print("쌍을 이루는 core 개수:", len(shared))

고유 core 개수: 2350
쌍을 이루는 core 개수: 74


In [6]:
# SMILES -> (y, w) 매핑 딕셔너리 미리 구성 (빠른 조회용)
smiles_to_label = {}
for i, s in enumerate(data['smiles_train'][:500]):  # 위에서 표본으로 쓴 것과 동일 범위
    smiles_to_label[s] = (data['y_train'][i], data['w_train'][i])

task_cols = data['task_cols']

def get_toxic_assays(smiles):
    """해당 분자가 어떤 assay에서 양성(독성)으로 측정됐는지 리스트 반환."""
    y, w = smiles_to_label.get(smiles, (None, None))
    if y is None:
        return None
    positive = [task_cols[i] for i in range(len(task_cols)) if w[i] == 1 and y[i] == 1]
    return positive


# 라벨이 갈리는(하나는 독성 있고 하나는 없는) 쌍 찾기
divergent_pairs = []
for core, members in shared.items():
    labels = [(smi, small, get_toxic_assays(smi)) for smi, small in members]
    # 독성 있는 것과 없는 것이 섞여있는지 확인
    has_toxic = any(l[2] for l in labels if l[2] is not None)
    has_clean = any(l[2] == [] for l in labels if l[2] is not None)
    if has_toxic and has_clean:
        divergent_pairs.append((core, labels))

print(f"라벨이 갈리는 쌍(core): {len(divergent_pairs)}개")
for core, labels in divergent_pairs[:5]:
    print(f"\ncore: {core}")
    for smi, small, assays in labels:
        print(f"  분자: {smi} | 치환부위: {small} | 독성assay: {assays}")

라벨이 갈리는 쌍(core): 31개

core: c1ccc([*:1])cc1
  분자: O=C(CS)Nc1ccccc1 | 치환부위: O=C(CS)N[*:1] | 독성assay: ['NR-AhR', 'NR-ER', 'SR-p53']
  분자: COC(=O)Cc1ccccc1 | 치환부위: COC(=O)C[*:1] | 독성assay: []
  분자: COC(=O)c1ccccc1 | 치환부위: COC(=O)[*:1] | 독성assay: []
  분자: NC(=O)Nc1ccccc1 | 치환부위: NC(=O)N[*:1] | 독성assay: []
  분자: OCCc1ccccc1 | 치환부위: OCC[*:1] | 독성assay: ['NR-AR-LBD']
  분자: CC(=O)CCc1ccccc1 | 치환부위: CC(=O)CC[*:1] | 독성assay: []
  분자: CNC(=O)c1ccccc1 | 치환부위: CNC(=O)[*:1] | 독성assay: ['SR-ARE']
  분자: CC(=O)NNc1ccccc1 | 치환부위: CC(=O)NN[*:1] | 독성assay: []

core: CC(C(=O)O)[*:1]
  분자: CC(O)C(=O)O | 치환부위: O[*:1] | 독성assay: ['NR-AR']
  분자: CC(C)C(=O)O | 치환부위: C[*:1] | 독성assay: []

core: CCCCCCCCCC[*:1]
  분자: CCCCCCCCCCCCCCCC[N+](C)(C)C | 치환부위: C[N+](C)(C)CCCCCC[*:1] | 독성assay: ['NR-ER', 'NR-ER-LBD', 'SR-MMP']
  분자: CCCCCCCCCCCCn1cc[n+](C)c1 | 치환부위: C[n+]1ccn(CC[*:1])c1 | 독성assay: ['SR-MMP']
  분자: CCCCCCCCCCCCC(=O)O | 치환부위: O=C(O)CC[*:1] | 독성assay: []
  분자: CCCCCCCCCCCCOS(=O)(=O)[O-] | 치환부위: O=S(=O)([O-])

In [7]:
def get_fragment_charge(fragment_smiles):
    """[*:1]을 더미 탄소로 치환한 뒤 formal charge 총합 계산."""
    mol = Chem.MolFromSmiles(fragment_smiles.replace('[*:1]', 'C'))
    if mol is None:
        return None
    return Chem.GetFormalCharge(mol)

# 테스트
print(get_fragment_charge("C[N+](C)(C)CCCCC[*:1]"))  # 양이온이니 +1이 나와야 정상
print(get_fragment_charge("O=C(O)CC[*:1]"))            # 카르복실산 중성형
print(get_fragment_charge("O[*:1]"))                   # 히드록실
print(get_fragment_charge("C[*:1]"))                    # 메틸

1
0
0
0


In [8]:
def find_reliable_substitution_pairs(divergent_pairs):
    """같은 core 안에서, 독성 있는 분자의 치환부위와 독성 없는 분자의 치환부위를
    전하가 일치하는 것끼리만 짝지어 반환."""
    reliable_pairs = []

    for core, labels in divergent_pairs:
        toxic_members = [(smi, small, assays) for smi, small, assays in labels if assays]
        clean_members = [(smi, small, assays) for smi, small, assays in labels if assays == []]

        for t_smi, t_small, t_assays in toxic_members:
            t_charge = get_fragment_charge(t_small)
            for c_smi, c_small, c_assays in clean_members:
                c_charge = get_fragment_charge(c_small)
                if t_charge is not None and t_charge == c_charge:
                    reliable_pairs.append({
                        "core": core,
                        "toxic_smiles": t_smi,
                        "toxic_fragment": t_small,
                        "toxic_assays": t_assays,
                        "clean_smiles": c_smi,
                        "clean_fragment": c_small,
                        "charge": t_charge,
                    })

    return reliable_pairs

reliable = find_reliable_substitution_pairs(divergent_pairs)
print(f"전하 일치하는 신뢰할 만한 치환쌍: {len(reliable)}개\n")
for r in reliable:
    print(f"[{r['toxic_assays']}] {r['toxic_fragment']}  →  {r['clean_fragment']}  (charge={r['charge']})")
    print(f"   원본: {r['toxic_smiles']}  →  {r['clean_smiles']}\n")

전하 일치하는 신뢰할 만한 치환쌍: 57개

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1]  →  COC(=O)C[*:1]  (charge=0)
   원본: O=C(CS)Nc1ccccc1  →  COC(=O)Cc1ccccc1

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1]  →  COC(=O)[*:1]  (charge=0)
   원본: O=C(CS)Nc1ccccc1  →  COC(=O)c1ccccc1

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1]  →  NC(=O)N[*:1]  (charge=0)
   원본: O=C(CS)Nc1ccccc1  →  NC(=O)Nc1ccccc1

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1]  →  CC(=O)CC[*:1]  (charge=0)
   원본: O=C(CS)Nc1ccccc1  →  CC(=O)CCc1ccccc1

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1]  →  CC(=O)NN[*:1]  (charge=0)
   원본: O=C(CS)Nc1ccccc1  →  CC(=O)NNc1ccccc1

[['NR-AR-LBD']] OCC[*:1]  →  COC(=O)C[*:1]  (charge=0)
   원본: OCCc1ccccc1  →  COC(=O)Cc1ccccc1

[['NR-AR-LBD']] OCC[*:1]  →  COC(=O)[*:1]  (charge=0)
   원본: OCCc1ccccc1  →  COC(=O)c1ccccc1

[['NR-AR-LBD']] OCC[*:1]  →  NC(=O)N[*:1]  (charge=0)
   원본: OCCc1ccccc1  →  NC(=O)Nc1ccccc1

[['NR-AR-LBD']] OCC[*:1]  →  CC(=O)CC[*:1]  (charge=0)
   원본: OCCc1ccccc1  →  CC(=O)C

In [9]:
from rdkit.Chem import Descriptors

def get_fragment_mw(fragment_smiles):
    mol = Chem.MolFromSmiles(fragment_smiles.replace('[*:1]', 'C'))
    if mol is None:
        return None
    return Descriptors.MolWt(mol)


def find_reliable_substitution_pairs(divergent_pairs, max_mw_diff=50.0):
    """전하가 같고, 분자량 차이가 max_mw_diff(Da) 이하인 치환쌍만 반환."""
    reliable_pairs = []

    for core, labels in divergent_pairs:
        toxic_members = [(smi, small, assays) for smi, small, assays in labels if assays]
        clean_members = [(smi, small, assays) for smi, small, assays in labels if assays == []]

        for t_smi, t_small, t_assays in toxic_members:
            t_charge = get_fragment_charge(t_small)
            t_mw = get_fragment_mw(t_small)
            for c_smi, c_small, c_assays in clean_members:
                c_charge = get_fragment_charge(c_small)
                c_mw = get_fragment_mw(c_small)
                if t_charge is None or c_charge is None or t_mw is None or c_mw is None:
                    continue
                if t_charge != c_charge:
                    continue
                if abs(t_mw - c_mw) > max_mw_diff:
                    continue
                reliable_pairs.append({
                    "core": core,
                    "toxic_smiles": t_smi, "toxic_fragment": t_small, "toxic_assays": t_assays,
                    "clean_smiles": c_smi, "clean_fragment": c_small,
                    "charge": t_charge, "mw_diff": abs(t_mw - c_mw),
                })

    return reliable_pairs

reliable2 = find_reliable_substitution_pairs(divergent_pairs, max_mw_diff=50.0)
print(f"전하+분자량 필터 통과한 치환쌍: {len(reliable2)}개\n")
for r in reliable2:
    print(f"[{r['toxic_assays']}] {r['toxic_fragment']} → {r['clean_fragment']} (Δmw={r['mw_diff']:.1f})")

전하+분자량 필터 통과한 치환쌍: 44개

[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1] → COC(=O)C[*:1] (Δmw=17.1)
[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1] → COC(=O)[*:1] (Δmw=31.1)
[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1] → NC(=O)N[*:1] (Δmw=31.1)
[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1] → CC(=O)CC[*:1] (Δmw=19.0)
[['NR-AhR', 'NR-ER', 'SR-p53']] O=C(CS)N[*:1] → CC(=O)NN[*:1] (Δmw=17.1)
[['NR-AR-LBD']] OCC[*:1] → COC(=O)C[*:1] (Δmw=28.0)
[['NR-AR-LBD']] OCC[*:1] → COC(=O)[*:1] (Δmw=14.0)
[['NR-AR-LBD']] OCC[*:1] → NC(=O)N[*:1] (Δmw=14.0)
[['NR-AR-LBD']] OCC[*:1] → CC(=O)CC[*:1] (Δmw=26.0)
[['NR-AR-LBD']] OCC[*:1] → CC(=O)NN[*:1] (Δmw=28.0)
[['SR-ARE']] CNC(=O)[*:1] → COC(=O)C[*:1] (Δmw=15.0)
[['SR-ARE']] CNC(=O)[*:1] → COC(=O)[*:1] (Δmw=1.0)
[['SR-ARE']] CNC(=O)[*:1] → NC(=O)N[*:1] (Δmw=1.0)
[['SR-ARE']] CNC(=O)[*:1] → CC(=O)CC[*:1] (Δmw=13.0)
[['SR-ARE']] CNC(=O)[*:1] → CC(=O)NN[*:1] (Δmw=15.0)
[['NR-AR']] O[*:1] → C[*:1] (Δmw=2.0)
[['NR-AR-LBD']] OC[*:1] → COC(=O)[*:1] (Δmw=28.0)
[['NR-

In [10]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H][c]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [11]:
import importlib
import src.tools.replacement_library
importlib.reload(src.tools.replacement_library)
from src.tools.replacement_library import get_replacement_candidates

print(get_replacement_candidates("phenol"))
print(get_replacement_candidates("amide"))

# 실제 분자로 확인
test_phenol = Chem.MolFromSmiles("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1")
print("phenol SMARTS 매치 확인:", test_phenol.HasSubstructMatch(Chem.MolFromSmarts("[OX2H][c]")))

{'problem_smarts': '[OX2H][c]', 'candidates': [{'smiles': 'Cl', 'name': 'chlorine', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 (quinone 형성 등) 경로를 차단하는 것으로 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}, {'smiles': 'C', 'name': 'methyl', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}]}
{'problem_smarts': '[NX3H1][CX3](=O)[#6]', 'candidates': [{'smiles': 'NC(=O)N', 'name': 'urea', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 계열(우레아 활용)로 일관성 있음', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}]}
phenol SMARTS 매치 확인: True


In [12]:
# phenol 규칙 실전 테스트
core_phenol = find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol")
print("phenol core_test:", core_phenol)

fix_phenol = propose_fix("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol", candidate_idx=0)
print("phenol fix_test:", fix_phenol)

# amide 규칙 실전 테스트
core_amide = find_core_and_target("CNC(=O)c1ccccc1", "amide")
print("\namide core_test:", core_amide)

fix_amide = propose_fix("CNC(=O)c1ccccc1", "amide", candidate_idx=0)
print("amide fix_test:", fix_amide)

phenol core_test: None
phenol fix_test: None

amide core_test: {'core': 'c1ccc([*:1])cc1', 'target_removed': 'CNC(=O)[*:1]'}
amide fix_test: {'new_smiles': 'NC(=O)Nc1ccccc1', 'candidate_used': 'urea', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 계열(우레아 활용)로 일관성 있음', 'is_valid': True}


In [13]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    # attachment point 개수만큼 더미 탄소가 붙었으니 그만큼 빼줘야 함
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---
    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            # core는 attachment point를 2개([*:1],[*:2]) 가짐.
            # target(=part)이 [*:1] 쪽인지 [*:2] 쪽인지 확인해서,
            # target이 아닌 쪽 attachment point에 other_chain_part를 먼저 이어붙여
            # 완전한(attachment point 1개만 남은) core를 만든다.
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            keep_ap = '[*:2]' if target_ap == '[*:1]' else '[*:1]'

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                # core의 keep_ap 자리에 other_chain_part를 결합
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            # merged 결과에는 target_ap가 [*:1] 라벨로 남아있어야 reassemble_molecule과 호환됨.
            # molzip 이후 라벨 번호가 바뀔 수 있으므로, 표준 [*:1] 하나만 남았는지 확인.
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                # 라벨이 [*:2]로 남았다면 [*:1]로 통일
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [14]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

# 회귀 테스트 (기존 잘 되던 것들)
print("=== 회귀 테스트 ===")
print("alkyl_halide:", propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print("aniline:", propose_fix("N#Cc1cc(N)ccc1NCCO", "aniline", candidate_idx=0))
print("amide:", propose_fix("CNC(=O)c1ccccc1", "amide", candidate_idx=0))

# 신규: phenol (Case B 필요)
print("\n=== phenol (신규, Case B) ===")
core_phenol2 = find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol")
print("core_test:", core_phenol2)
fix_phenol2 = propose_fix("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol", candidate_idx=0)
print("fix_test:", fix_phenol2)

=== 회귀 테스트 ===
alkyl_halide: {'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
aniline: {'new_smiles': 'N#Cc1cc(C(N)=O)ccc1NCCO', 'candidate_used': 'acetamide (acylated amine)', 'rationale': '1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단', 'is_valid': True}
amide: {'new_smiles': 'NC(=O)Nc1ccccc1', 'candidate_used': 'urea', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 계열(우레아 활용)로 일관성 있음', 'is_valid': True}

=== phenol (신규, Case B) ===
core_test: None
fix_test: None


In [15]:
from rdkit.Chem import rdMMPA

mol_phenol = Chem.MolFromSmiles("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1")
fragments_phenol = rdMMPA.FragmentMol(mol_phenol, maxCuts=2, resultsAsMols=False)
for core, chain in fragments_phenol:
    print(f"core: {core}  |  chain: {chain}")

core:   |  chain: O=C(c1ccc(Cl)cc1)[*:1].Oc1ccc([*:1])cc1
core: c1cc([*:2])ccc1[*:1]  |  chain: O=C(c1ccc(Cl)cc1)[*:1].O[*:2]
core: O=C([*:1])[*:2]  |  chain: Clc1ccc([*:1])cc1.Oc1ccc([*:2])cc1
core: O=C(c1ccc([*:1])cc1)[*:2]  |  chain: Cl[*:1].Oc1ccc([*:2])cc1
core:   |  chain: O=C(c1ccc(Cl)cc1)c1ccc([*:1])cc1.O[*:1]
core: O=C(c1ccc([*:1])cc1)[*:2]  |  chain: Clc1ccc([*:2])cc1.O[*:1]
core: O=C(c1ccc([*:1])cc1)c1ccc([*:2])cc1  |  chain: Cl[*:1].O[*:2]
core:   |  chain: Clc1ccc([*:1])cc1.O=C(c1ccc(O)cc1)[*:1]
core: c1cc([*:2])ccc1[*:1]  |  chain: Cl[*:2].O=C(c1ccc(O)cc1)[*:1]
core:   |  chain: Cl[*:1].O=C(c1ccc(O)cc1)c1ccc([*:1])cc1


In [ ]:
test_o = Chem.MolFromSmiles("OC")  # O[*:1]을 C로 치환한 것
pattern_oh = Chem.MolFromSmarts("[OX2H]")
print("매치 여부:", test_o.HasSubstructMatch(pattern_oh))
print("heavy atoms:", test_o.GetNumHeavyAtoms())

매치 여부: True
heavy atoms: 2


In [16]:
core_test = Chem.MolFromSmiles("O=C(c1ccc([*:1])cc1)[*:2]")
other_test = Chem.MolFromSmiles("Clc1ccc([*:2])cc1")

print("core_test:", core_test)
print("other_test:", other_test)

try:
    merged_test = Chem.molzip(core_test, other_test)
    print("merged:", Chem.MolToSmiles(merged_test))
except Exception as e:
    print("에러:", e)

core_test: <rdkit.Chem.rdchem.Mol object at 0x7caec1525bd0>
other_test: <rdkit.Chem.rdchem.Mol object at 0x7caee3221a80>
merged: O=C(c1ccc(Cl)cc1)c1ccc([*:1])cc1


[10:53:36] Incomplete atom labelling, cannot make bond


In [17]:
mol = Chem.MolFromSmiles("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1")
problem_pattern = Chem.MolFromSmarts("[OX2H]")
pattern_size = problem_pattern.GetNumAtoms()

fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
for core, chain in fragments2:
    if not core:
        continue
    chain_parts = chain.split('.')
    if len(chain_parts) != 2:
        continue
    for i, part in enumerate(chain_parts):
        part_mol = Chem.MolFromSmiles(part.replace('[*:1]', 'C').replace('[*:2]', 'C'))
        if part_mol is None:
            print(f"파싱실패: {part}")
            continue
        matched = part_mol.HasSubstructMatch(problem_pattern)
        n_att = part.count('[*:')
        size_ok = (part_mol.GetNumHeavyAtoms() - n_att == pattern_size)
        print(f"core={core} | part={part} | matched={matched} | size_ok={size_ok}")

core=c1cc([*:2])ccc1[*:1] | part=O=C(c1ccc(Cl)cc1)[*:1] | matched=False | size_ok=False
core=c1cc([*:2])ccc1[*:1] | part=O[*:2] | matched=True | size_ok=True
core=O=C([*:1])[*:2] | part=Clc1ccc([*:1])cc1 | matched=False | size_ok=False
core=O=C([*:1])[*:2] | part=Oc1ccc([*:2])cc1 | matched=True | size_ok=False
core=O=C(c1ccc([*:1])cc1)[*:2] | part=Cl[*:1] | matched=False | size_ok=True
core=O=C(c1ccc([*:1])cc1)[*:2] | part=Oc1ccc([*:2])cc1 | matched=True | size_ok=False
core=O=C(c1ccc([*:1])cc1)[*:2] | part=Clc1ccc([*:2])cc1 | matched=False | size_ok=False
core=O=C(c1ccc([*:1])cc1)[*:2] | part=O[*:1] | matched=True | size_ok=True
core=O=C(c1ccc([*:1])cc1)c1ccc([*:2])cc1 | part=Cl[*:1] | matched=False | size_ok=True
core=O=C(c1ccc([*:1])cc1)c1ccc([*:2])cc1 | part=O[*:2] | matched=True | size_ok=True
core=c1cc([*:2])ccc1[*:1] | part=Cl[*:2] | matched=False | size_ok=True
core=c1cc([*:2])ccc1[*:1] | part=O=C(c1ccc(O)cc1)[*:1] | matched=True | size_ok=False


In [18]:
core_test2 = Chem.MolFromSmiles("c1cc([*:2])ccc1[*:1]")
other_test2 = Chem.MolFromSmiles("O=C(c1ccc(Cl)cc1)[*:1]")

try:
    merged_test2 = Chem.molzip(core_test2, other_test2)
    print("merged:", Chem.MolToSmiles(merged_test2))
except Exception as e:
    print("에러:", e)

merged: O=C(c1ccc(Cl)cc1)c1ccc([*:2])cc1


[10:53:39] Incomplete atom labelling, cannot make bond


In [19]:
mol = Chem.MolFromSmiles("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1")
problem_pattern = Chem.MolFromSmarts("[OX2H]")
pattern_size = problem_pattern.GetNumAtoms()

fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
for core, chain in fragments2:
    if not core:
        continue
    chain_parts = chain.split('.')
    if len(chain_parts) != 2:
        continue
    for i, part in enumerate(chain_parts):
        part_mol = Chem.MolFromSmiles(part.replace('[*:1]', 'C').replace('[*:2]', 'C'))
        if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
            continue
        n_att = part.count('[*:')
        if part_mol.GetNumHeavyAtoms() - n_att != pattern_size:
            continue

        print(f"매치 성공! core={core}, part={part}")
        other_chain_part = chain_parts[1 - i]
        print(f"  other_chain_part={other_chain_part}")

        target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
        print(f"  target_ap={target_ap}")

        core_mol = Chem.MolFromSmiles(core)
        other_mol = Chem.MolFromSmiles(other_chain_part)
        print(f"  core_mol is None: {core_mol is None}, other_mol is None: {other_mol is None}")

        try:
            merged = Chem.molzip(core_mol, other_mol)
            merged_smiles = Chem.MolToSmiles(merged)
            print(f"  merged_smiles={merged_smiles}, count[*:]={merged_smiles.count('[*:')}")
        except Exception as e:
            print(f"  molzip 예외: {e}")
        print("---")

매치 성공! core=c1cc([*:2])ccc1[*:1], part=O[*:2]
  other_chain_part=O=C(c1ccc(Cl)cc1)[*:1]
  target_ap=[*:2]
  core_mol is None: False, other_mol is None: False
  merged_smiles=O=C(c1ccc(Cl)cc1)c1ccc([*:2])cc1, count[*:]=1
---
매치 성공! core=O=C(c1ccc([*:1])cc1)[*:2], part=O[*:1]
  other_chain_part=Clc1ccc([*:2])cc1
  target_ap=[*:1]
  core_mol is None: False, other_mol is None: False
  merged_smiles=O=C(c1ccc(Cl)cc1)c1ccc([*:1])cc1, count[*:]=1
---
매치 성공! core=O=C(c1ccc([*:1])cc1)c1ccc([*:2])cc1, part=O[*:2]
  other_chain_part=Cl[*:1]
  target_ap=[*:2]
  core_mol is None: False, other_mol is None: False
  merged_smiles=O=C(c1ccc(Cl)cc1)c1ccc([*:2])cc1, count[*:]=1
---


[10:53:41] Incomplete atom labelling, cannot make bond
[10:53:41] Incomplete atom labelling, cannot make bond
[10:53:41] Incomplete atom labelling, cannot make bond


In [20]:
import inspect
print(inspect.getsource(find_core_and_target))

def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---

In [22]:
import inspect
print(inspect.getsource(src.tools.molecule_editor._check_and_match))

result_direct = find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol")
print("직접 호출 결과:", result_direct)

# Case A만 따로 확인
fragments1_check = rdMMPA.FragmentMol(Chem.MolFromSmiles("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1"), maxCuts=1, resultsAsMols=False)
for core, chain in fragments1_check:
    if core:
        continue
    parts = chain.split('.')
    if len(parts) != 2:
        continue
    for i, part in enumerate(parts):
        from src.tools.molecule_editor import _check_and_match
        matched = _check_and_match(part, problem_pattern, pattern_size)
        print(f"Case A: part={part}, matched={matched}")

def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    # attachment point 개수만큼 더미 탄소가 붙었으니 그만큼 빼줘야 함
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size

직접 호출 결과: None
Case A: part=O=C(c1ccc(Cl)cc1)[*:1], matched=False
Case A: part=Oc1ccc([*:1])cc1, matched=False
Case A: part=O=C(c1ccc(Cl)cc1)c1ccc([*:1])cc1, matched=False
Case A: part=O[*:1], matched=True
Case A: part=Clc1ccc([*:1])cc1, matched=False
Case A: part=O=C(c1ccc(O)cc1)[*:1], matched=False
Case A: part=Cl[*:1], matched=False
Case A: part=O=C(c1ccc(O)cc1)c1ccc([*:1])cc1, matched=False


In [23]:
try:
    result_direct2 = find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol")
    print("결과:", result_direct2)
except Exception as e:
    import traceback
    traceback.print_exc()

결과: None


In [24]:
import src.tools.replacement_library
print(src.tools.replacement_library.get_replacement_candidates("phenol"))

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target

print(src.tools.replacement_library.get_replacement_candidates("phenol"))
print(find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol"))

{'problem_smarts': '[OX2H][c]', 'candidates': [{'smiles': 'Cl', 'name': 'chlorine', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 (quinone 형성 등) 경로를 차단하는 것으로 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}, {'smiles': 'C', 'name': 'methyl', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}]}
{'problem_smarts': '[OX2H][c]', 'candidates': [{'smiles': 'Cl', 'name': 'chlorine', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 (quinone 형성 등) 경로를 차단하는 것으로 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}, {'smiles': 'C', 'name': 'methyl', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정', 'source': 'data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)'}]}
None


In [25]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [26]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

print(find_core_and_target("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol"))
print(propose_fix("O=C(c1ccc(O)cc1)c1ccc(Cl)cc1", "phenol", candidate_idx=0))

{'core': 'O=C(c1ccc(Cl)cc1)c1ccc([*:1])cc1', 'target_removed': 'O[*:1]'}
{'new_smiles': 'O=C(c1ccc(Cl)cc1)c1ccc(Cl)cc1', 'candidate_used': 'chlorine', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 (quinone 형성 등) 경로를 차단하는 것으로 추정', 'is_valid': True}


In [5]:
!pip install -U google-genai -q


In [6]:
from google import genai

gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)
print("Gemini 클라이언트 준비 완료")

Gemini 클라이언트 준비 완료


In [8]:
multi_known_test = None
for s in data['smiles_train'][:2000]:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_known_test = s
        break

print("찾은 분자:", multi_known_test)

찾은 분자: Nc1ccc(NCCO)c([N+](=O)[O-])c1


In [9]:
result_regression = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("최종 상태:", result_regression['status'])
for h in result_regression['history']:
    print(h)

최종 상태: success
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]}
{'step': 1, 'smiles': 'N#Cc1cc(N)ccc1NCCO', 'fixed_rule': 'nitro_group', 'problem_reason': '니트로기는 Ames 유전독성 및 변이원성을 유발하는 대표적인 고위험 toxicophore이므로, 안전성 확보를 위해 우선적으로 치환해야 합니다.', 'candidate_used': 'nitrile', 'candidate_reason': '니트로기의 강한 전자 끌림 특성을 유지하여 주변 아민의 산화를 방지하고 분자의 전자적 성질을 보존하면서도, 유전독성 유발 위험을 효과적으로 배제할 수 있는 가장 검증된 대체기입니다.', 'problems': [{'rule_name': 'aniline', 'atom_indices': [2, 3, 4, 5, 6, 7, 8]}]}
{'step': 2, 'smiles': 'N#Cc1cc(F)ccc1NCCO', 'fixed_rule': 'aniline', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'fluorine', 'candidate_reason': '반응성이 높은 1차 아닐린 기를 불소로 대체하여 아닐린 특유의 유전독성(Ames mutagenicity) 유발 가능성을 원천적으로 배제하고, 전자끄는 효과를 통해 남아있는 아민의 산화 반응성도 낮출 수 있기 때문입니다.', 'problems': []}


In [10]:
!git add -A
!git commit -m "Final verification: Case B fragmentation + full agent loop regression test pass"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 36d923f] Final verification: Case B fragmentation + full agent loop regression test pass
 2 files changed, 85 insertions(+), 13 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.18 KiB | 2.18 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   d447e91..36d923f  main -> main
